# 06 — Baselines de prédiction du churn

Construire une première référence reproductible, comprendre ses erreurs et préparer les comparaisons futures.
**Dataset principal uniquement.** Kernel : **Python (Telco Churn)**.

Les scores portent sur des snapshots, pas des clients indépendants. La continuation du panel autorise
les clients communs entre périodes. L'horizon de cible, la disponibilité des features et la maturité des labels
restent non confirmés ; le test 2024 a déjà été exploré. Ce notebook produit une **baseline de développement**.
Aucun score ne certifie un usage en production ni l'effet causal d'une campagne.

## 1. Reprendre les partitions auditées

**Question.** Quelles données le modèle peut-il utiliser ?

Nous relisons les exports du notebook 05 et vérifions leur cohérence avec le découpage recalculé.
Seules les features du manifeste entrent dans X : identifiant, date, cible et `source_row` restent exclus.
Un contrôle de fuite en échec bloque l'entraînement ; les points non confirmés sont affichés.

In [1]:
import json
import platform
import warnings

import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sklearn
from IPython.display import display
from sklearn.calibration import calibration_curve
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import precision_recall_curve

from telco_churn.settings import Settings
from telco_churn.modeling.temporal_split import TemporalConfig, TemporalSplitter, load_processed
from telco_churn.modeling.baseline import build_baselines, evaluate, select_threshold

warnings.filterwarnings("error", category=ConvergenceWarning)
settings = Settings.load()
config = TemporalConfig.load(settings)
source, source_manifest = load_processed(settings)
splitter = TemporalSplitter(config, settings.churn_target_column)
result = splitter.split(source)
manifest = json.loads((settings.split_output_dir / "split_manifest.json").read_text())
if manifest["config"] != config.model_dump(mode="json"):
    raise ValueError("Configuration modifiée : réexécuter le notebook 05.")
features = source_manifest["feature_columns"]
target = settings.churn_target_column
if manifest["feature_columns"] != features or manifest["failed_checks"]:
    raise ValueError("Manifeste incohérent ou contrôles de fuite en échec.")
for name, frame in result.partitions.items():
    exported = pd.read_csv(settings.split_output_dir / f"{name}.csv",
                           dtype=source_manifest.get("feature_dtypes", {}))
    expected = frame.reset_index().copy()
    expected[config.date_column] = expected[config.date_column].dt.strftime(config.date_format)
    pd.testing.assert_frame_equal(exported, expected, check_dtype=False)
leakage = splitter.leakage_report(result, source_manifest)
if leakage.status.eq("FAIL").any():
    raise ValueError("Contrôle de fuite en échec.")
display(leakage.loc[leakage.status.isin(["WARN", "UNVERIFIED"])])
display(result.summary(config, target))
train = result.partitions["train"]
validation = result.partitions["validation"]
test = result.partitions["test"]
output_dir = settings.project_root / "reports" / "baseline"
output_dir.mkdir(parents=True, exist_ok=True)


,check,status,detail
9,label_horizon,UNVERIFIED,30 jours : hypothèse
10,feature_availability,UNVERIFIED,Sources de 17 features non confirmées ; voir c...
12,untouched_test,WARN,Période déjà explorée dans l'EDA : test de dév...
13,test_label_maturity,UNVERIFIED,Fin de disponibilité des labels non documentée


,partition,n_rows,n_clients,first_date,last_date,n_positive_snapshots,n_positive_clients,positive_percent
0,train,20103,3762,2024-01-31,2024-06-30,590,590,2.934885
1,validation,6309,3239,2024-08-31,2024-09-30,209,209,3.312728
2,test,5910,3003,2024-11-30,2024-12-31,165,165,2.791878


## 2. Définir des références simples

**Question.** Le modèle apporte-t-il plus qu'une probabilité constante ?

- `dummy_prior` prédit pour chaque ligne la fréquence du churn observée sur son train.
- La régression logistique apprend un score avec régularisation L2 par défaut (`C=1`), sans recherche d'hyperparamètres.

Chaque pipeline impute les numériques par leur médiane, les standardise et encode les catégories en one-hot.
Les catégories inconnues sont ignorées. Toutes ces opérations sont ajustées **uniquement sur le train du pipeline**.
Les colonnes entièrement manquantes sont conservées. Aucun suréchantillonnage ni poids de classe ne modifie
la prévalence ; un éventuel besoin de rappel sera traité par le seuil.

Avec environ 3 % de positifs, l'accuracy serait trompeuse. La métrique principale est l'**average precision (AP)**,
complétée par ROC AUC, Brier et log-loss. Brier et log-loss sont meilleurs quand ils sont faibles.
L'AP de la référence constante est égale à la prévalence de la période évaluée.

## 3. Vérifier la stabilité dans les backtests du train

**Question.** La baseline fonctionne-t-elle dans plusieurs périodes antérieures ?

Les folds restent dans le train principal, avec purge. On recrée et réajuste chaque pipeline dans chaque fold.
Les métriques au seuil 0,5 sont descriptives : aucun seuil n'est optimisé sur ces validations ni sur le test.

In [2]:
backtest_rows = []
for fold, partitions in splitter.backtests(result).items():
    fold_train, fold_valid = partitions["train"], partitions["validation"]
    for name, pipeline in build_baselines(fold_train[features]).items():
        pipeline.fit(fold_train[features], fold_train[target])
        probability = pipeline.predict_proba(fold_valid[features])[:, 1]
        backtest_rows.append({"fold": fold, "model": name,
                              **evaluate(fold_valid[target], probability)})
backtest_scores = pd.DataFrame(backtest_rows)
display(backtest_scores)
backtest_scores.to_csv(output_dir / "backtest_metrics.csv", index=False)

,fold,model,n_rows,prevalence,average_precision,roc_auc,brier,log_loss,threshold,precision,recall,f1,selected_fraction,tn,fp,fn,tp
0,fold_1,dummy_prior,3299,0.025765,0.025765,0.500000,0.025137,0.120310,0.5,0.0,0.000000,0.000000,0.000000,3214,0,85,0
1,fold_1,logistic_regression,3299,0.025765,0.101682,0.744815,0.024403,0.109668,0.5,0.0,0.000000,0.000000,0.000606,3212,2,85,0
2,fold_2,dummy_prior,3250,0.024000,0.024000,0.500000,0.023463,0.113931,0.5,0.0,0.000000,0.000000,0.000000,3172,0,78,0
3,fold_2,logistic_regression,3250,0.024000,0.140314,0.742814,0.022365,0.102672,0.5,1.0,0.012821,0.025316,0.000308,3172,0,77,1


## 4. Entraîner sur le train et comparer sur validation

**Question.** Quel modèle classe le mieux les futurs snapshots ?

Les deux modèles sont ajustés sur janvier–juin uniquement. Le candidat est choisi par AP sur validation ;
en cas d'égalité exacte, la référence naïve reste prioritaire. Le test ne participe pas au choix.

In [3]:
models = build_baselines(train[features])
validation_probabilities = {}
validation_rows = []
for name, pipeline in models.items():
    pipeline.fit(train[features], train[target])
    probability = pipeline.predict_proba(validation[features])[:, 1]
    validation_probabilities[name] = probability
    validation_rows.append({"model": name, **evaluate(validation[target], probability)})
validation_scores = pd.DataFrame(validation_rows)
display(validation_scores)
winner = validation_scores.sort_values("average_precision", ascending=False, kind="stable").iloc[0]["model"]
print("Candidat retenu sur validation :", winner)
fig = go.Figure()
for name, probability in validation_probabilities.items():
    precision, recall, _ = precision_recall_curve(validation[target], probability)
    fig.add_trace(go.Scatter(x=recall, y=precision, name=name, mode="lines"))
fig.add_hline(y=validation[target].mean(), line_dash="dash")
fig.update_layout(title="Validation — précision selon le rappel", xaxis_title="Rappel",
                  yaxis_title="Précision", template="plotly_white")
fig.show(renderer="plotly_mimetype")
fig.write_html(output_dir / "validation_precision_recall.html", include_plotlyjs=True)

,model,n_rows,prevalence,average_precision,roc_auc,brier,log_loss,threshold,precision,recall,f1,selected_fraction,tn,fp,fn,tp
0,dummy_prior,6309,0.033127,0.033127,0.50000,0.032044,0.145691,0.5,0.000000,0.000000,0.000000,0.000000,6100,0,209,0
1,logistic_regression,6309,0.033127,0.138518,0.78824,0.030417,0.127175,0.5,0.333333,0.004785,0.009434,0.000476,6098,2,208,1


Candidat retenu sur validation : logistic_regression


## 5. Choisir un seuil sur validation

**Question.** À partir de quelle probabilité signaler un snapshot ?

Le seuil 0,5 est une référence. Sans coûts métier confirmés, nous choisissons un second seuil maximisant F1
sur validation : compromis descriptif entre précision et rappel, **pas un optimum économique**.
En cas d'égalité, le seuil le plus élevé est retenu. Cette optimisation rend le F1 de validation optimiste.
La fraction signalée mesure le volume potentiel ; elle ne représente pas une liste de clients à contacter.
Le seuil et le modèle sont ensuite figés avant de calculer les scores du test.

In [4]:
threshold = select_threshold(validation[target], validation_probabilities[winner])
threshold_comparison = pd.DataFrame([
    {"rule": rule, **evaluate(validation[target], validation_probabilities[winner], value)}
    for rule, value in [("reference_0.5", 0.5), ("validation_f1", threshold)]
])
display(threshold_comparison)
print(f"Seuil retenu : {threshold:.6f}")

,rule,n_rows,prevalence,average_precision,roc_auc,brier,log_loss,threshold,precision,recall,f1,selected_fraction,tn,fp,fn,tp
0,reference_0.5,6309,0.033127,0.138518,0.78824,0.030417,0.127175,0.500000,0.333333,0.004785,0.009434,0.000476,6098,2,208,1
1,validation_f1,6309,0.033127,0.138518,0.78824,0.030417,0.127175,0.066497,0.155140,0.397129,0.223118,0.084799,5648,452,126,83


Seuil retenu : 0.066497


## 6. Évaluer sur le test de développement

**Question.** Que donnent les choix figés sur novembre–décembre ?

Aucun refit sur train + validation : cela changerait le score et pourrait invalider le seuil sélectionné.
Le test est évalué au seuil fixé, avec la référence constante pour comparaison. Il ne sert à aucun réglage.
Les résultats ne sont pas ceux d'un test final aveugle : cette période a déjà servi à l'EDA.

In [5]:
test_probability = models[winner].predict_proba(test[features])[:, 1]
dummy_probability = models["dummy_prior"].predict_proba(test[features])[:, 1]
test_scores = pd.DataFrame([
    {"model": winner, "rule": "validation_f1", **evaluate(test[target], test_probability, threshold)},
    {"model": winner, "rule": "reference_0.5", **evaluate(test[target], test_probability, 0.5)},
    {"model": "dummy_prior", "rule": "reference_0.5", **evaluate(test[target], dummy_probability, 0.5)},
])
display(test_scores)
selected = test_scores.iloc[0]
print(f"Au seuil figé : {int(selected.tp)} vrais positifs, {int(selected.fp)} faux positifs, "
      f"{int(selected.fn)} churns manqués, {int(selected.tn)} vrais négatifs.")

,model,rule,n_rows,prevalence,average_precision,roc_auc,brier,log_loss,threshold,precision,recall,f1,selected_fraction,tn,fp,fn,tp
0,logistic_regression,validation_f1,5910,0.027919,0.129067,0.789701,0.025842,0.111295,0.066497,0.142061,0.309091,0.194656,0.060745,5437,308,114,51
1,logistic_regression,reference_0.5,5910,0.027919,0.129067,0.789701,0.025842,0.111295,0.500000,0.000000,0.000000,0.000000,0.000000,5745,0,165,0
2,dummy_prior,reference_0.5,5910,0.027919,0.027919,0.500000,0.027141,0.127468,0.500000,0.000000,0.000000,0.000000,0.000000,5745,0,165,0


Au seuil figé : 51 vrais positifs, 308 faux positifs, 114 churns manqués, 5437 vrais négatifs.


## 7. Examiner les probabilités et les erreurs

**Question.** Le classement et les probabilités racontent-ils la même chose ?

La calibration compare fréquence observée et probabilité moyenne par groupe quantile.
Des groupes peuvent fusionner si les scores sont identiques. Cette courbe est descriptive :
aucun calibrateur n'est appris ici. Les scores Brier/log-loss complètent le graphique.
Les résultats mensuels révèlent une éventuelle instabilité ; ils ne servent pas à retoucher le modèle.
Les snapshots répétés d'un même client sont corrélés : nous ne calculons pas d'intervalle supposant des lignes indépendantes.

In [6]:
fig = go.Figure()
for name, frame, probability in [
    ("validation", validation, validation_probabilities[winner]),
    ("test", test, test_probability),
]:
    observed, predicted = calibration_curve(frame[target], probability, n_bins=8, strategy="quantile")
    fig.add_trace(go.Scatter(x=predicted, y=observed, mode="lines+markers", name=name))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Calibration idéale",
                         line=dict(dash="dash")))
fig.update_layout(title="Calibration descriptive du candidat", xaxis_title="Probabilité moyenne",
                  yaxis_title="Fréquence observée", template="plotly_white")
fig.show(renderer="plotly_mimetype")
fig.write_html(output_dir / "calibration.html", include_plotlyjs=True)
monthly_rows = []
for month, frame in test.assign(month=test[config.date_column].dt.to_period("M")).groupby("month"):
    if frame[target].nunique() == 2:
        probability = models[winner].predict_proba(frame[features])[:, 1]
        monthly_rows.append({"month": str(month), **evaluate(frame[target], probability, threshold)})
    else:
        print(f"{month} : métriques de classement non calculées, une seule classe.")
monthly_scores = pd.DataFrame(monthly_rows)
display(monthly_scores)

,month,n_rows,prevalence,average_precision,roc_auc,brier,log_loss,threshold,precision,recall,f1,selected_fraction,tn,fp,fn,tp
0,2024-11,3003,0.031968,0.145749,0.791419,0.029375,0.123754,0.066497,0.167539,0.333333,0.222997,0.063603,2748,159,64,32
1,2024-12,2907,0.023736,0.114549,0.786459,0.022192,0.098424,0.066497,0.113095,0.275362,0.160338,0.057792,2689,149,50,19


## 8. Sauvegarder une baseline reproductible

Le pipeline contient l'imputation, l'encodage, la standardisation et le modèle. Le manifeste conserve features,
seuil, versions, paramètres et limites du split. Les prédictions gardent `source_row` pour l'audit.
Ces artefacts sont destinés au développement ; ils ne déclenchent aucune campagne.
Le fichier joblib doit être chargé uniquement depuis une source de confiance.

In [7]:
validation_scores.to_csv(output_dir / "validation_metrics.csv", index=False)
threshold_comparison.to_csv(output_dir / "threshold_comparison.csv", index=False)
test_scores.to_csv(output_dir / "test_metrics.csv", index=False)
monthly_scores.to_csv(output_dir / "test_monthly_metrics.csv", index=False)
for name, frame, probability in [
    ("validation", validation, validation_probabilities[winner]),
    ("test", test, test_probability),
]:
    predictions = frame[[config.id_column, config.date_column, target]].copy()
    predictions["probability"] = probability
    predictions["predicted_churn"] = (probability >= threshold).astype(int)
    predictions.to_csv(output_dir / f"{name}_predictions.csv", index_label="source_row")
model_dir = settings.project_root / "models" / "baseline"
model_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(models[winner], model_dir / "pipeline.joblib")
payload = {
    "status": "provisional_development_baseline", "training_validated": False,
    "model": winner, "threshold": threshold, "threshold_rule": "max_validation_f1",
    "selection_metric": "validation_average_precision", "feature_columns": features,
    "target": target, "split_manifest": manifest,
    "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__,
                 "pandas": pd.__version__, "numpy": np.__version__},
    "model_parameters": models[winner].named_steps["model"].get_params(),
}
(model_dir / "manifest.json").write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n")
print("Rapports :", output_dir)
print("Pipeline et manifeste :", model_dir)
print(f"AP validation : {validation_scores.set_index('model').loc[winner, 'average_precision']:.4f}")
print(f"AP test de développement : {selected.average_precision:.4f}")

Rapports : /Users/surelmanda/2-LLops-Databricks-Projects/Telco-Churn-Optimization/reports/baseline
Pipeline et manifeste : /Users/surelmanda/2-LLops-Databricks-Projects/Telco-Churn-Optimization/models/baseline
AP validation : 0.1385
AP test de développement : 0.1291


### Lecture des résultats de cette exécution

La régression logistique dépasse la référence constante dans les deux backtests : AP **0,102** et **0,140**,
contre des prévalences de **2,58 %** et **2,40 %**. Cette variabilité rappelle que les périodes ne sont pas équivalentes.
Sur validation, son AP atteint **0,139**, contre **0,033** pour la référence. Elle est donc retenue.

Sur le test de développement, l'AP est de **0,129** contre **0,028** pour la référence ; le ROC AUC est de **0,790**.
Le Brier passe de **0,0271** à **0,0258** : une amélioration moyenne des probabilités, qui ne suffit pas à garantir
leur calibration dans chaque sous-groupe.

Au seuil de **0,06650** choisi sur validation, **359 snapshots** sont signalés : **51 vrais positifs et 308 faux positifs**.
Le rappel est de **30,9 %**, la précision de **14,2 %** et **114 snapshots positifs restent manqués**.
Le seuil 0,5 ne détecte aucun positif sur ce test. Le seuil sélectionné permet donc de détecter davantage de churn,
avec un volume d'erreurs qui doit être examiné selon les coûts métier. Ces résultats ne prouvent pas qu'une offre
empêcherait le churn des personnes signalées.

Ces chiffres décrivent les exports actuels ; après un changement des données ou de configuration,
les tableaux recalculés font foi.

## 9. Interpréter et préparer la suite

Comparer l'AP à la référence constante et examiner précision, rappel, volume signalé et churns manqués ensemble.
Un bon ROC AUC ne suffit pas avec une cible rare. Un F1 élevé n'établit pas la rentabilité d'une campagne.
Les résultats des cellules précédentes sont recalculés à chaque exécution ; les effectifs ne sont pas codés en dur.

Avant d'aller plus loin : confirmer la définition temporelle de la cible et les sources disponibles au snapshot,
examiner les erreurs et la stabilité des backtests, puis comparer les futurs modèles avec le même protocole.
La calibration et les seuils économiques devront utiliser des données dédiées ou une validation temporelle adaptée.
Une période encore non explorée reste nécessaire pour une évaluation finale. La probabilité de churn ne mesure
pas l'effet d'une offre de rétention.